### 导入库和自定义函数

In [27]:
import warnings
warnings.filterwarnings('ignore')
%matplotlib inline

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.special import jn
from IPython.display import display, clear_output
from tqdm import tqdm
from sklearn import linear_model
from sklearn import preprocessing
from sklearn.svm import SVR
from sklearn.ensemble import RandomForestRegressor,GradientBoostingRegressor
from sklearn.decomposition import PCA,FastICA,FactorAnalysis,SparsePCA
from sklearn.model_selection import GridSearchCV,cross_val_score,StratifiedKFold,train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error
from itertools import combinations
from keras.layers import Conv1D, Activation, MaxPool1D, Flatten, Dense
from keras.layers import Input, Dense, Concatenate, Reshape, Dropout,  Add
from keras.layers import Dropout
from keras.callbacks import Callback, EarlyStopping
from keras.callbacks import LearningRateScheduler
from keras import backend as K
import tensorflow as tf


In [28]:
from data_preprocessing import reduce_mem_usage,process_outliers_and_issues
from feature_engineering import  compute_category_counts,generate_cross_category_numeric_features,generate_cross_category_features,apply_target_encoding_with_cv,TargetMeanEncoder


### 数据处理

In [29]:
## 读取数据
train = reduce_mem_usage(pd.read_csv('used_car_train_20200313.csv', sep=' '))
test = reduce_mem_usage(pd.read_csv('used_car_testB_20200421.csv', sep=' '))
# 使用示例
train = process_outliers_and_issues(train)
test = process_outliers_and_issues(test)

Memory usage of dataframe is 37200132.00 MB
Memory usage after optimization is: 10200264.00 MB
Decreased by 72.6%
Memory usage of dataframe is 12000132.00 MB
Memory usage after optimization is: 3200264.00 MB
Decreased by 73.3%


In [30]:
concat_data = pd.concat([train,test ])

### 特征工程

In [31]:
from itertools import combinations

# 加法组合
for i, j in combinations(['v_' + str(i) for i in range(15)], 2):
    concat_data[f'{i}+{j}'] = concat_data[i] + concat_data[j]  # 数量C(15,2)=105

# 生成两两组合
for col1, col2 in combinations(['v_' + str(i) for i in range(15)], 2):
    concat_data[f'{col1}*{col2}'] = concat_data[col1] * concat_data[col2]
for i in ['model','brand', 'bodyType', 'fuelType','gearbox', 'power', 'kilometer', 'notRepairedDamage', 'regionCode']:
    for j in ['v_' +str(i) for i in range(15)]:
        concat_data[str(i)+'*'+str(j)] = concat_data[i]*concat_data[j]    
concat_data.shape

(200000, 376)

In [32]:
# 使用时间：data['creatDate'] - data['regDate']
data = concat_data.copy()
data['used_time1_days'] = (pd.to_datetime(data['creatDate'], format='%Y-%m-%d', errors='coerce') - 
                         pd.to_datetime(data['regDate'], format='%Y-%m-%d', errors='coerce')).dt.days

data['used_time2_days'] = (pd.to_datetime('2020-01-01', format='%Y-%m-%d', errors='coerce') - 
                         pd.to_datetime(data['regDate'], format='%Y-%m-%d', errors='coerce')).dt.days

data['used_time3_days'] = (pd.to_datetime('2020-01-01', format='%Y-%m-%d', errors='coerce') - 
                         pd.to_datetime(data['creatDate'], format='%Y-%m-%d', errors='coerce')).dt.days

# 转换为月份（天数/30）
data['used_time1_months'] = data['used_time1_days'] / 30
data['used_time2_months'] = data['used_time2_days'] / 30
data['used_time3_months'] = data['used_time3_days'] / 30

# 转换为年份（天数/365）
data['used_time1_years'] = data['used_time1_days'] / 365
data['used_time2_years'] = data['used_time2_days'] / 365
data['used_time3_years'] = data['used_time3_days'] / 365

In [33]:
# 计算类别特征的计数
category_columns = ['model', 'brand', 'regionCode', 'bodyType', 'fuelType', 'name', 'regDate', 'creatDate', 'kilometer']
data = compute_category_counts(data, category_columns)

# 用数值特征对类别特征做统计刻画
cross_category_columns = ['model', 'brand', 'bodyType', 'fuelType']
numeric_columns = ['v_0', 'v_3', 'v_8', 'v_11', 'v_12', 'power', 'kilometer']
data = generate_cross_category_numeric_features(data, numeric_columns, cross_category_columns)

# 生成类别与类别之间的交叉特征
data = generate_cross_category_features(data)


100%|██████████| 3/3 [00:08<00:00,  2.76s/it]


In [34]:
## 选择特征列
numerical_cols = data.columns

cat_fea = ['SaleID','offerType','seller']
feature_cols = [col for col in numerical_cols if col not in cat_fea]
feature_cols = [col for col in feature_cols if col not in ['price']]

## 构造训练样本和测试样本
X_data = data.iloc[:len(train),:][feature_cols]
Y_data = train['price']
X_test  = data.iloc[len(train):,:][feature_cols]
X_data['price'] = train['price']

In [35]:
categorical_cols = ['model', 'brand', 'name', 'regionCode', 'regDate', 'creatDate']
encoder = TargetMeanEncoder(categorical_cols, target_type='regression')

# 训练集
X_data = encoder.fit_transform(X_data, Y_data)

# 测试集
X_test = encoder.transform(X_test)

# 保持目标列
X_data['price'] = train['price']


In [36]:
# 调用目标编码函数
X_data, X_test = apply_target_encoding_with_cv(X_data, Y_data, X_test, ['regionCode', 'brand', 'kilometer', 'model'])
drop_list = ['regDate', 'creatDate']
x_train = X_data.drop(drop_list+['price'],axis=1)
x_test = X_test.drop(drop_list,axis=1)
x_train = x_train.astype('float32')
x_test = x_test.astype('float32')
x_test

100%|██████████| 4/4 [00:12<00:00,  3.19s/it]


,name,model,brand,bodyType,fuelType,gearbox,power,kilometer,notRepairedDamage,regionCode,...,kilometer_target_median,kilometer_target_max,kilometer_target_min,kilometer_target_sum,model_target_mean,model_target_std,model_target_median,model_target_max,model_target_min,model_target_sum
150000,133777.0,67.0,0.0,1.0,0.0,0.0,101.0,15.0,0.0,5019.0,...,2261.000000,99999.0,11.0,333540288.0,1461.614624,783.265320,1334.849976,7405.000000,18.500000,1.425975e+06
150001,61206.0,19.0,6.0,2.0,0.0,0.0,73.0,6.0,0.0,1505.0,...,9519.000000,93041.0,22.6,40906016.0,6738.362305,9888.337891,2944.000000,98409.500000,20.000000,5.805540e+07
150002,67829.0,5.0,5.0,4.0,0.0,0.0,120.0,5.0,0.0,1776.0,...,10761.900391,99531.0,50.5,38415128.0,3236.637695,2635.700684,2500.000000,19768.099609,26.000000,6.009523e+06
150003,8892.0,22.0,9.0,1.0,0.0,0.0,58.0,15.0,0.0,26.0,...,2261.000000,99999.0,11.0,333540288.0,2233.526855,2434.965820,1453.000000,24659.000000,57.500000,3.063462e+06
150004,76998.0,46.0,6.0,0.0,0.0,0.0,116.0,15.0,0.0,738.0,...,2261.000000,99999.0,11.0,333540288.0,3975.416748,4677.651855,2191.750000,38010.000000,22.000000,8.779909e+06
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
199995,111443.0,4.0,4.0,0.0,0.0,1.0,150.0,15.0,0.0,5564.0,...,2261.000000,99999.0,11.0,333540288.0,6198.346191,6335.432617,3899.899902,94679.101562,47.299999,4.711071e+07
199996,152834.0,65.0,1.0,0.0,0.0,0.0,179.0,4.0,0.0,5220.0,...,12186.400391,93360.0,105.0,36891500.0,7901.725098,7045.652832,5869.899902,45265.000000,41.000000,1.941456e+07
199997,132531.0,4.0,4.0,0.0,0.0,1.0,147.0,12.5,0.0,3795.0,...,4119.750000,87060.0,16.4,90064040.0,6198.346191,6335.432617,3899.899902,94679.101562,47.299999,4.711071e+07
199998,143405.0,40.0,1.0,4.0,0.0,1.0,176.0,15.0,0.0,61.0,...,2261.000000,99999.0,11.0,333540288.0,6818.097656,6340.463379,4910.399902,46600.000000,17.400000,2.762598e+07


### pca 降维

In [39]:
from sklearn.preprocessing import MinMaxScaler,RobustScaler, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn import decomposition


# 1. 初始化预处理对象
#scaler = RobustScaler() 效果非常差 
#scaler =StandardScaler() 一般般
scaler = MinMaxScaler()
imputer = SimpleImputer(strategy='most_frequent')  # or 'median', 'most_frequent','mean'，KNNImputer,都可以试试
pca = decomposition.PCA(n_components=200)

# 2. 只在训练集上 fit 和 transform（学习参数并应用）
x_train_processed = scaler.fit_transform(x_train.values)
x_train_processed = imputer.fit_transform(x_train_processed)
x_train_pca = pca.fit_transform(x_train_processed)

# 3. 在测试集上仅 transform（复用训练集的参数）
x_test_processed = scaler.transform(x_test.values)  
x_test_processed = imputer.transform(x_test_processed)     
x_test_pca = pca.transform(x_test_processed)                

X_pca = x_train_pca 
y = train['price'].values

In [40]:
# 显示每个主成分的方差占比
print("每个主成分的方差占比:", pca.explained_variance_ratio_)

# 显示累计方差占比
print("累计方差占比:", np.cumsum(pca.explained_variance_ratio_))


每个主成分的方差占比: [1.98819727e-01 1.36734575e-01 1.09493539e-01 8.74766707e-02
 5.93111478e-02 5.21063097e-02 4.98409644e-02 3.81232202e-02
 3.47022563e-02 2.53651571e-02 2.04818193e-02 1.94636974e-02
 1.74159463e-02 1.29059218e-02 1.05887307e-02 6.67036464e-03
 6.18290296e-03 6.04839437e-03 5.64191723e-03 5.17722731e-03
 5.13508962e-03 4.96863108e-03 4.59826179e-03 4.41363361e-03
 4.06519976e-03 3.80696030e-03 3.29291727e-03 2.96637602e-03
 2.75830040e-03 2.66715721e-03 2.49705394e-03 2.41763401e-03
 2.17829132e-03 2.03698175e-03 1.99149572e-03 1.84779463e-03
 1.74562936e-03 1.69525726e-03 1.60121708e-03 1.53531670e-03
 1.40125304e-03 1.38524291e-03 1.35100342e-03 1.23419531e-03
 1.19904382e-03 1.14811014e-03 1.06113323e-03 1.04548095e-03
 1.03394128e-03 9.88947111e-04 9.22835257e-04 8.92780488e-04
 8.73093610e-04 8.38796666e-04 8.10301513e-04 7.79770606e-04
 7.64066877e-04 7.36083777e-04 6.92818139e-04 6.42191560e-04
 6.26591791e-04 6.18174556e-04 6.04965724e-04 5.62716392e-04
 5.31516038e

### pytroch

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import KFold
from sklearn.metrics import mean_absolute_error
from torch.utils.data import DataLoader, TensorDataset


class Attention(nn.Module): 
    def __init__(self, input_dim, reduction_ratio=8):
        super(Attention, self).__init__()
        self.fc1 = nn.Linear(input_dim, input_dim // reduction_ratio)
        self.fc2 = nn.Linear(input_dim // reduction_ratio, input_dim)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        attention = self.fc1(x)
        attention = torch.relu(attention)
        attention = self.fc2(attention)
        attention = self.sigmoid(attention)
        return x * attention  # Attention加权输出


# 添加BatchNorm层 + Dropout
class NNModelWithAttention(nn.Module):
    def __init__(self, input_dim, dropout_rate=0.2):
        super(NNModelWithAttention, self).__init__()
        self.dense1 = nn.Linear(input_dim, 256)
        self.bn1 = nn.BatchNorm1d(256)
        self.attention = Attention(256)  
        self.dropout1 = nn.Dropout(dropout_rate)

        self.dense2 = nn.Linear(256, 256)
        self.bn2 = nn.BatchNorm1d(256)
        self.dropout2 = nn.Dropout(dropout_rate)

        self.dense3 = nn.Linear(256, 64)
        self.dense4 = nn.Linear(64, 32)
        self.dense5 = nn.Linear(32, 8)

        self.out = nn.Linear(8, 1)

    def forward(self, x):
        x = torch.relu(self.bn1(self.dense1(x)))
        x = self.attention(x)  
        x = self.dropout1(x)

        x = torch.relu(self.bn2(self.dense2(x)))
        x = self.dropout2(x)

        x = torch.relu(self.dense3(x))
        x = torch.relu(self.dense4(x))
        x = torch.relu(self.dense5(x))


        return self.out(x)



In [ ]:
import torch
import torch.optim as optim
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import mean_absolute_error
import pandas as pd
import numpy as np
from sklearn.model_selection import KFold
from torch.optim.lr_scheduler import StepLR

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

n_splits = 8
kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)

b_size = 2048
max_epochs = 120
oof_pred = np.zeros(len(X_pca))  # 训练集 OOF
sub = pd.read_csv('used_car_testB_20200421.csv', sep=' ')[['SaleID']].copy()
sub['price'] = 0

avg_mae = 0
trained_fold_models = []  # 保存每个 fold 的模型用于测试集预测

for fold, (trn_idx, val_idx) in enumerate(kf.split(X_pca, y)):
    print(f'fold: {fold}')
    
    # 分割数据集
    X_train, y_train = X_pca[trn_idx], y[trn_idx]
    X_val, y_val = X_pca[val_idx], y[val_idx]

    # 转换为 GPU 张量
    X_train_tensor = torch.tensor(X_train, dtype=torch.float32).to(device)
    y_train_tensor = torch.tensor(y_train, dtype=torch.float32).to(device)
    X_val_tensor = torch.tensor(X_val, dtype=torch.float32).to(device)
    y_val_tensor = torch.tensor(y_val, dtype=torch.float32).to(device)

    train_loader = DataLoader(TensorDataset(X_train_tensor, y_train_tensor), batch_size=b_size, shuffle=True)
    val_loader = DataLoader(TensorDataset(X_val_tensor, y_val_tensor), batch_size=b_size, shuffle=False)

    # 模型 + 优化器 + 学习率调度
    model = NNModelWithAttention(X_train.shape[1]).to(device)
    optimizer = optim.Adam(model.parameters(), lr=0.015)
    scheduler = StepLR(optimizer, step_size=12, gamma=0.5)
    criterion = nn.L1Loss()

    early_stopping_counter = 0
    best_val_mae = float('inf')
    best_y_pred = np.zeros(len(y_val))

    # ---------- 训练 ----------
    for epoch in range(max_epochs):
        model.train()
        running_loss = 0.0
        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels.view(-1, 1))
            loss.backward()
            optimizer.step()
            running_loss += loss.item()
        train_mae = running_loss / len(train_loader)

        model.eval()
        val_loss = 0.0
        y_pred = np.zeros(len(y_val))
        with torch.no_grad():
            for i, (inputs, labels) in enumerate(val_loader):
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = model(inputs)
                loss = criterion(outputs, labels.view(-1, 1))
                val_loss += loss.item()
                y_pred[i * b_size:(i + 1) * b_size] = outputs.cpu().numpy().reshape(-1)
        val_mae = val_loss / len(val_loader)

        if val_mae < best_val_mae:
            best_val_mae = val_mae
            best_y_pred = y_pred.copy()
            early_stopping_counter = 0
        else:
            early_stopping_counter += 1
            if early_stopping_counter > 25:
                print("提前停止触发")
                break

        scheduler.step()

        print(f"Epoch {epoch}/{max_epochs}, train_mae: {train_mae:.4f}, val_mae: {val_mae:.4f}")

    # 保存 OOF
    oof_pred[val_idx] = best_y_pred
    avg_mae += best_val_mae / n_splits
    print(f'fold {fold} 的最佳 val_mae: {best_val_mae:.4f}')

    # 保存 fold 模型用于测试集预测
    trained_fold_models.append(model)

# ---------- 训练集 OOF MAE ----------
final_mae = mean_absolute_error(y, oof_pred)
print(f"最终的 out-of-fold MAE: {final_mae:.4f}")





Using device: cuda
fold: 0
Epoch 0/120, train_mae: 2416.0426, val_mae: 755.4561
Epoch 1/120, train_mae: 855.6700, val_mae: 720.0875
Epoch 2/120, train_mae: 793.1898, val_mae: 612.8764
Epoch 3/120, train_mae: 730.6349, val_mae: 614.0131
Epoch 4/120, train_mae: 736.5494, val_mae: 577.6943
Epoch 5/120, train_mae: 723.8247, val_mae: 539.8598
Epoch 6/120, train_mae: 721.6222, val_mae: 648.6590
Epoch 7/120, train_mae: 705.2976, val_mae: 599.5877
Epoch 8/120, train_mae: 698.5010, val_mae: 504.5869
Epoch 9/120, train_mae: 665.0318, val_mae: 536.1674
Epoch 10/120, train_mae: 675.3977, val_mae: 533.1159
Epoch 11/120, train_mae: 660.5753, val_mae: 571.1461
Epoch 12/120, train_mae: 636.8518, val_mae: 769.6782
Epoch 13/120, train_mae: 627.0397, val_mae: 479.0429
Epoch 14/120, train_mae: 610.4208, val_mae: 495.0516
Epoch 15/120, train_mae: 620.7072, val_mae: 535.3952
Epoch 16/120, train_mae: 630.4595, val_mae: 508.1557
Epoch 17/120, train_mae: 647.8313, val_mae: 511.4761
Epoch 18/120, train_mae: 642

In [ ]:
# ---------- 测试集预测 ----------
X_test_tensor = torch.tensor(x_test_pca, dtype=torch.float32).to(device)
test_dataset = TensorDataset(X_test_tensor)
test_loader = DataLoader(test_dataset, batch_size=b_size, shuffle=False)

test_preds = np.zeros(len(x_test_pca))
for fold_model in trained_fold_models:
    fold_model.eval()
    fold_preds = []
    with torch.no_grad():
        for inputs in test_loader:
            inputs = inputs[0].to(device)
            outputs = fold_model(inputs)
            fold_preds.append(outputs.cpu().numpy())
    test_preds += np.concatenate(fold_preds).ravel()  # <- squeeze 成一维


test_preds /= n_splits
sub['price'] = test_preds
sub.to_csv('submission.csv', index=False)
print("测试集预测已保存到 submission.csv")